In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
import pandas as pd
from tqdm import tqdm
from urllib.parse import urlparse
import json
from google.colab import files

In [ ]:
BASE_URL = 'https://db.chgk.info'

In [ ]:
tournaments_data = []
authors_data = []
questions_data = []

author_counter = 0
author_map = {}

# Ссылки на турниры

In [ ]:
def get_tournament_links():
    """Собирает все ссылки на турниры со всех страниц"""
    tournament_links = []

    for page in tqdm(range(0, 277), desc="Сбор ссылок"):
        url = f'{BASE_URL}/last?page={page}'

        try:
            response = requests.get(url, timeout=15)
            response.encoding = 'utf-8'

            if response.status_code != 200:
                print(f"Ошибка на странице {page}: статус {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, 'html.parser')
            table = soup.find('table', class_='last_packages')

            if not table:
                continue

            rows = table.find_all('tr')
            for row in rows:
                if row.find('th'):
                    continue

                cells = row.find_all('td')
                if len(cells) >= 1:
                    first_cell = cells[0]
                    link = first_cell.find('a')

                    if link and link.get('href'):
                        href = link.get('href')
                        if href.startswith('/tour/'):
                            full_link = BASE_URL + href
                            if full_link not in tournament_links:
                                tournament_links.append(full_link)

            time.sleep(0.3)

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")
            continue

    print(f"Найдено {len(tournament_links)} уникальных турниров")
    return tournament_links

# Парсинг страницы турнира

In [19]:
def parse_tournament_page(url, tournament_id):
    """Парсит одну страницу турнира и возвращает данные"""
    global author_counter, author_map

    try:
        response = requests.get(url, timeout=30)
        response.encoding = 'utf-8'

        if response.status_code != 200:
            print(f"Ошибка доступа к {url}: {response.status_code}")
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        # название турнира
        title_elem = soup.find('h1', class_='title')
        if not title_elem:
            print(f"Не найдено название турнира: {url}")
            return None
        title = title_elem.get_text(strip=True)

        # дата
        date_elem = soup.find('strong', string='Дата:')
        date = ''
        if date_elem:
            date_parent = date_elem.parent
            if date_parent:
                date_text = date_parent.get_text(strip=True)
                date = date_text.replace('Дата:', '').strip()

        # турнир
        tournament = {
            'id': tournament_id,
            'title': title,
            'date': date
        }

        # вопросы
        questions = []
        question_blocks = soup.find_all('div', class_='question')

        for q_block in question_blocks:
            if q_block.find('div', class_='razdatka'):
                continue

            # текст вопроса
            question_text_elem = q_block.find('strong', class_='Question')
            if not question_text_elem:
                continue

            question_elem = q_block.find('strong', class_='Question')
            if not question_elem:
                continue
            parent_p = question_elem.find_parent('p')

            question_text = ''
            if parent_p:
                p_clone = BeautifulSoup(str(parent_p), 'html.parser')

                for q_tag in p_clone.find_all('strong', class_='Question'):
                    q_tag.decompose()

                question_text = p_clone.get_text(strip=True)
                question_text = re.sub(r'Вопрос\s+\d+:', '', question_text).strip()

            # ответ
            answer_elem = q_block.find('strong', class_='Answer')
            answer = answer_elem.parent.get_text(strip=True).replace('Ответ:', '').strip() if answer_elem else ''

            # зачет
            pass_criteria_elem = q_block.find('strong', class_='PassCriteria')
            pass_criteria = pass_criteria_elem.parent.get_text(strip=True).replace('Зачёт:', '').strip() if pass_criteria_elem else ''

            # комментарий
            comment_elem = q_block.find('strong', class_='Comments')
            comment = comment_elem.parent.get_text(strip=True).replace('Комментарий:', '').strip() if comment_elem else ''

            # автор
            author_elem = q_block.find('strong', class_='Authors')
            author_names = []
            author_ids = []

            if author_elem:
                author_text = author_elem.parent.get_text(strip=True)
                author_text = re.sub(r'Автор(?:ы)?:', '', author_text).strip()

                author_text = author_text.replace('&', ',')
                parts = re.split(r',\s*|\s+и\s+', author_text)

                for part in parts:
                    part = part.strip().replace('\xa0', ' ').replace('\u200c', '').replace('.', '')
                    part = re.sub(r'\s*\([^)]*\)', '', part)
                    part = ' '.join(part.split())
                    part = part.rstrip(',')
                    if part:
                        author_names.append(part)

                        if part not in author_map:
                            author_counter += 1
                            author_map[part] = author_counter
                            authors_data.append({
                                'id': author_counter,
                                'name': part
                            })
                        author_ids.append(author_map[part])

            question_text = ' '.join(question_text.split()).replace('\xa0', ' ').replace('\u200c', '')
            answer = ' '.join(answer.split()).replace('\xa0', ' ').replace('\u200c', '')
            pass_criteria = ' '.join(pass_criteria.split()).replace('\xa0', ' ').replace('\u200c', '')
            comment = ' '.join(comment.split()).replace('\xa0', ' ').replace('\u200c', '')

            question = {
                'question_text': question_text,
                'answer': answer,
                'pass_criteria': pass_criteria,
                'comment': comment,
                'author_ids': author_ids,
                'tournament_id': tournament_id,
            }

            questions.append(question)

        return tournament, questions

    except Exception as e:
        print(f"Ошибка при парсинге {url}: {e}")
        return None

# Сохранение данных

In [ ]:
import pickle

In [ ]:
def save_data():
    data = {
        'tournaments': tournaments_data,
        'authors': authors_data,
        'questions': questions_data,
        'author_map': author_map
    }

    with open('chgk_data.pkl', 'wb') as f:
        pickle.dump(data, f)

    pd.DataFrame(tournaments_data).to_csv('tournaments.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(authors_data).to_csv('authors.csv', index=False, encoding='utf-8-sig')

    df = pd.DataFrame(questions_data)
    df['author_ids'] = df['author_ids'].apply(lambda x: ','.join(map(str, x)) if x else '')
    df.to_csv('questions.csv', index=False, encoding='utf-8-sig')

# Процесс парсинга

In [ ]:
import os

In [ ]:
tournament_links = get_tournament_links()

Сбор ссылок: 100%|██████████| 277/277 [07:46<00:00,  1.69s/it]

Найдено 5502 уникальных турниров


In [ ]:
with open('tournament_links.txt', 'w', encoding='utf-8') as file:
    for link in tournament_links:
        file.write(link + '\n')

In [ ]:
def init_parsing(offset=0, limit=500):
    global tournaments_data, questions_data, authors_data, author_map, author_counter

    if os.path.exists('chgk_data.pkl'):
        with open('chgk_data.pkl', 'rb') as f:
            data = pickle.load(f)
        tournaments_data = data.get('tournaments', [])
        questions_data = data.get('questions', [])
        authors_data = data.get('authors', [])
        author_map = data.get('author_map', {})
        author_counter = len(author_map)

    with open('tournament_links.txt', 'r', encoding='utf-8') as file:
        all_links = [line.strip() for line in file.readlines() if line.strip()]

    selected = all_links[offset:offset + limit]

    for idx, link in enumerate(tqdm(selected, desc="Парсинг")):
        result = parse_tournament_page(link, len(tournaments_data) + idx + 1)
        if result:
            tournament, questions = result
            tournaments_data.append(tournament)
            questions_data.extend(questions)

        time.sleep(0.3)

        if (idx + 1) % 100 == 0:
            save_data()

    save_data()
    return tournaments_data, questions_data

In [ ]:
init_parsing(offset=0, limit=500)

Парсинг:  66%|██████▌   | 331/500 [11:48<07:07,  2.53s/it]

Ошибка при парсинге https://db.chgk.info/tour/okmar22_u: ('Connection broken: IncompleteRead(3908 bytes read, 4276 more expected)', IncompleteRead(3908 bytes read, 4276 more expected))


Парсинг:  93%|█████████▎| 466/500 [18:15<01:40,  2.95s/it]

Ошибка при парсинге https://db.chgk.info/tour/okmar19_u: ('Connection broken: IncompleteRead(7627 bytes read, 2613 more expected)', IncompleteRead(7627 bytes read, 2613 more expected))


Парсинг:  95%|█████████▌| 477/500 [18:53<01:08,  2.98s/it]

Ошибка при парсинге https://db.chgk.info/tour/okmar20_u: ('Connection broken: IncompleteRead(3531 bytes read, 6709 more expected)', IncompleteRead(3531 bytes read, 6709 more expected))


Парсинг: 100%|██████████| 500/500 [20:04<00:00,  2.41s/it]


([{'id': 5,
   'title': 'Синхронный турнир «Что-то вроде „В конце ноября“»',
   'date': ''},
  {'id': 2,
   'title': 'Соловушка. Беседа #6, с Денисом, Илоной, Ильёй и Рузелем',
   'date': ''},
  {'id': 6,
   'title': 'IX открытый школьный фестиваль по интеллектуальным играм «Интеллектуальный воронёнок»',
   'date': '2023-11-26'},
  {'id': 1, 'title': 'Коперник', 'date': ''},
  {'id': 8,
   'title': 'VIII Международный синхронный турнир ДРУГОЙ СТАРЫЙ СВЕТ (ДСС-21).',
   'date': '2021-12-18'},
  {'id': 9,
   'title': 'Открытый синхронный турнир "Квизобрейн-3"',
   'date': '2023-06-11'},
  {'id': 4, 'title': 'Кнопочки', 'date': ''},
  {'id': 11,
   'title': 'Чизкейкфест. Пролог / Синхронный турнир «Чизкейк»',
   'date': '17.04.2022 - 14.05.2022'},
  {'id': 3,
   'title': 'Соловушка. Беседа #4, с Владимиром, Иваном и Максимом',
   'date': '2024-06-02'},
  {'id': 10,
   'title': 'Синхронный турнир «Части Вселенной 1960–1999» (2022)',
   'date': '27.03.2022 - 26.04.2022'},
  {'id': 7,
   'ti

In [ ]:
def repeat_parsing():
    if os.path.exists('chgk_data.pkl'):
        with open('chgk_data.pkl', 'rb') as f:
            data = pickle.load(f)
        start = len(data.get('tournaments', []))
        print(f"Уже спарсено: {start} турниров")

        tournaments_data, questions_data = init_parsing(offset=start, limit=500)

In [ ]:
repeat_parsing()

Уже спарсено: 778 турниров


Парсинг:  89%|████████▉ | 445/500 [19:52<04:47,  5.22s/it]

Ошибка при парсинге https://db.chgk.info/tour/okmar18: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


Парсинг: 100%|██████████| 500/500 [22:39<00:00,  2.72s/it]


In [ ]:
repeat_parsing()

Уже спарсено: 1277 турниров


Парсинг:   3%|▎         | 16/500 [00:46<20:12,  2.51s/it]

Ошибка при парсинге https://db.chgk.info/tour/izmi1617: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


Парсинг:   3%|▎         | 17/500 [01:01<52:02,  6.46s/it]

Ошибка при парсинге https://db.chgk.info/tour/rostov18: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


Парсинг:  49%|████▉     | 244/500 [13:31<15:07,  3.54s/it]

Ошибка при парсинге https://db.chgk.info/tour/okmar17: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


Парсинг: 100%|██████████| 500/500 [27:37<00:00,  3.32s/it]


In [ ]:
def load_existing_data():
    global tournaments_data, questions_data, authors_data, author_map, author_counter


    if os.path.exists('chgk_data.pkl'):
        with open('chgk_data.pkl', 'rb') as f:
            data = pickle.load(f)
        tournaments_data = data.get('tournaments', [])
        questions_data = data.get('questions', [])
        authors_data = data.get('authors', [])
        author_map = data.get('author_map', {})
        author_counter = len(author_map)
        print(f"Загружено: {len(tournaments_data)} турниров, {len(questions_data)} вопросов")
        return True

    elif os.path.exists('tournaments.csv') and os.path.exists('questions.csv'):
        df_tournaments = pd.read_csv('tournaments.csv')
        df_questions = pd.read_csv('questions.csv')

        tournaments_data = df_tournaments.to_dict('records')

        questions_data = []
        for _, row in df_questions.iterrows():
            q = row.to_dict()
            q['author_ids'] = [int(x) for x in str(q['author_ids']).split(',') if x] if pd.notna(q['author_ids']) else []
            questions_data.append(q)

        if os.path.exists('authors.csv'):
            df_authors = pd.read_csv('authors.csv')
            authors_data = df_authors.to_dict('records')
            author_map = {a['name']: a['id'] for a in authors_data}
            author_counter = len(author_map)

        print(f"Загружено из CSV: {len(tournaments_data)} турниров, {len(questions_data)} вопросов")
        return True

    return False

In [ ]:
def continue_parsing(limit=500):
    global tournaments_data, questions_data, authors_data, author_map, author_counter

    if not load_existing_data():
        print("Нет существующих данных. Начинаем с нуля.")

    with open('tournament_links.txt', 'r', encoding='utf-8') as file:
        all_links = [line.strip() for line in file.readlines() if line.strip()]

    start = len(tournaments_data)
    total_links = len(all_links)

    if start >= total_links:
        print(f"Все {total_links} турниров уже спарсены!")
        return

    print(f"Уже спарсено: {start} из {total_links}")
    print(f"Добавляем еще {min(limit, total_links - start)} турниров")

    selected = all_links[start:start + limit]

    for idx, link in enumerate(tqdm(selected, desc="Парсинг")):
        result = parse_tournament_page(link, start + idx + 1)
        if result:
            tournament, questions = result
            tournaments_data.append(tournament)
            questions_data.extend(questions)

        time.sleep(0.3)

        if (idx + 1) % 100 == 0:
            save_data()
            print(f"Сохранено: {len(tournaments_data)} турниров")

    save_data()
    print(f"\nГотово! Всего: {len(tournaments_data)} турниров, {len(questions_data)} вопросов")

In [ ]:
continue_parsing(limit=500)

Загружено: 1774 турниров, 79131 вопросов
Уже спарсено: 1774 из 5502
Добавляем еще 500 турниров


Парсинг:   3%|▎         | 17/500 [00:56<39:06,  4.86s/it]

Ошибка при парсинге https://db.chgk.info/tour/okmar16: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


Парсинг:  11%|█         | 55/500 [02:56<18:33,  2.50s/it]

Ошибка при парсинге https://db.chgk.info/tour/knitu14: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


...

Парсинг:  81%|████████  | 406/500 [20:58<06:49,  4.36s/it]

Ошибка при парсинге https://db.chgk.info/tour/cup13brj: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


Парсинг: 100%|██████████| 500/500 [26:00<00:00,  3.12s/it]

Сохранено: 2269 турниров



Готово! Всего: 2269 турниров, 105307 вопросов


In [ ]:
continue_parsing(limit=500)

Загружено: 2269 турниров, 105307 вопросов
Уже спарсено: 2269 из 5502
Добавляем еще 500 турниров


Парсинг:   3%|▎         | 14/500 [00:34<25:17,  3.12s/it]

Ошибка при парсинге https://db.chgk.info/tour/russv07: ('Connection broken: IncompleteRead(7996 bytes read, 196 more expected)', IncompleteRead(7996 bytes read, 196 more expected))


Парсинг:   5%|▌         | 25/500 [01:09<35:21,  4.47s/it]

Ошибка при парсинге https://db.chgk.info/tour/okmar12: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


...

Парсинг:  97%|█████████▋| 487/500 [27:38<00:59,  4.55s/it]

Ошибка при парсинге https://db.chgk.info/tour/kol10v: ('Connection broken: IncompleteRead(3531 bytes read, 6709 more expected)', IncompleteRead(3531 bytes read, 6709 more expected))


Парсинг: 100%|██████████| 500/500 [28:34<00:00,  3.43s/it]

Сохранено: 2756 турниров



Готово! Всего: 2756 турниров, 133184 вопросов


In [ ]:
continue_parsing(limit=500)

Загружено: 2756 турниров, 133184 вопросов
Уже спарсено: 2756 из 5502
Добавляем еще 500 турниров


Парсинг:   3%|▎         | 13/500 [00:28<13:01,  1.60s/it]

Ошибка при парсинге https://db.chgk.info/tour/arhang12: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


Парсинг:   3%|▎         | 16/500 [01:05<1:07:18,  8.34s/it]

Ошибка при парсинге https://db.chgk.info/tour/gele09br: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


...

Парсинг:  87%|████████▋ | 434/500 [27:51<03:25,  3.11s/it]

Ошибка при парсинге https://db.chgk.info/tour/ukbr10sh: ('Connection broken: IncompleteRead(9675 bytes read, 565 more expected)', IncompleteRead(9675 bytes read, 565 more expected))


Парсинг: 100%|██████████| 500/500 [30:41<00:00,  3.68s/it]

Сохранено: 3229 турниров



Готово! Всего: 3229 турниров, 163130 вопросов


In [ ]:
continue_parsing(limit=500)

Загружено: 3229 турниров, 163130 вопросов
Уже спарсено: 3229 из 5502
Добавляем еще 500 турниров


Парсинг:  20%|██        | 100/500 [05:47<50:08,  7.52s/it]

Сохранено: 3329 турниров


Парсинг:  40%|████      | 200/500 [10:15<18:19,  3.67s/it]

Сохранено: 3429 турниров


...

Парсинг:  80%|████████  | 400/500 [19:16<19:57, 11.97s/it]

Сохранено: 3627 турниров


Парсинг: 100%|██████████| 500/500 [24:25<00:00,  2.93s/it]

Сохранено: 3727 турниров



Готово! Всего: 3727 турниров, 197255 вопросов


In [ ]:
continue_parsing(limit=500)

Загружено: 3727 турниров, 197255 вопросов
Уже спарсено: 3727 из 5502
Добавляем еще 500 турниров


Парсинг:  20%|██        | 100/500 [04:10<34:14,  5.14s/it]

Сохранено: 3827 турниров


Парсинг:  25%|██▌       | 126/500 [05:31<27:57,  4.49s/it]

Ошибка при парсинге https://db.chgk.info/tour/kol0506: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


...

Парсинг:  95%|█████████▌| 476/500 [27:37<01:04,  2.68s/it]

Ошибка при парсинге https://db.chgk.info/tour/dneprop: ('Connection broken: IncompleteRead(9675 bytes read, 565 more expected)', IncompleteRead(9675 bytes read, 565 more expected))


Парсинг: 100%|██████████| 500/500 [28:49<00:00,  3.46s/it]

Сохранено: 4204 турниров



Готово! Всего: 4204 турниров, 227947 вопросов


In [ ]:
continue_parsing(limit=500)

Загружено: 4204 турниров, 227947 вопросов
Уже спарсено: 4204 из 5502
Добавляем еще 500 турниров


Парсинг:  14%|█▎        | 68/500 [02:56<17:26,  2.42s/it]

Ошибка при парсинге https://db.chgk.info/tour/marafo03: HTTPSConnectionPool(host='db.chgk.info', port=443): Read timed out. (read timeout=15)


Парсинг:  18%|█▊        | 91/500 [04:10<15:26,  2.26s/it]

Ошибка при парсинге https://db.chgk.info/tour/haif1l04: ('Connection broken: IncompleteRead(9675 bytes read, 565 more expected)', IncompleteRead(9675 bytes read, 565 more expected))


...

Парсинг:  91%|█████████ | 454/500 [23:16<02:20,  3.06s/it]

Ошибка при парсинге https://db.chgk.info/tour/burda92: ('Connection broken: IncompleteRead(3908 bytes read, 4276 more expected)', IncompleteRead(3908 bytes read, 4276 more expected))


Парсинг: 100%|██████████| 500/500 [25:13<00:00,  3.03s/it]

Сохранено: 4689 турниров



Готово! Всего: 4689 турниров, 262816 вопросов


In [ ]:
continue_parsing(limit=500)

Загружено: 4689 турниров, 262816 вопросов
Уже спарсено: 4689 из 5502
Добавляем еще 500 турниров


Парсинг:  15%|█▌        | 77/500 [02:28<14:25,  2.05s/it]

Ошибка при парсинге https://db.chgk.info/tour/ukrbr02: ('Connection broken: IncompleteRead(9675 bytes read, 565 more expected)', IncompleteRead(9675 bytes read, 565 more expected))


Парсинг:  17%|█▋        | 84/500 [02:52<18:01,  2.60s/it]

Ошибка при парсинге https://db.chgk.info/tour/ukbr01sh: ('Connection broken: IncompleteRead(9675 bytes read, 565 more expected)', IncompleteRead(9675 bytes read, 565 more expected))


...

Парсинг:  80%|████████  | 400/500 [18:56<09:13,  5.54s/it]

Сохранено: 5080 турниров


Парсинг: 100%|██████████| 500/500 [23:30<00:00,  2.82s/it]

Сохранено: 5180 турниров



Готово! Всего: 5180 турниров, 302140 вопросов


In [ ]:
continue_parsing(limit=500)

Загружено: 5180 турниров, 302140 вопросов
Уже спарсено: 5180 из 5502
Добавляем еще 322 турниров


Парсинг:   4%|▍         | 13/322 [00:28<16:34,  3.22s/it]

Ошибка при парсинге https://db.chgk.info/tour/mgp0001: ('Connection broken: IncompleteRead(3531 bytes read, 6709 more expected)', IncompleteRead(3531 bytes read, 6709 more expected))


Парсинг:  15%|█▍        | 48/322 [01:46<12:24,  2.72s/it]

Ошибка при парсинге https://db.chgk.info/tour/mgp9900: ('Connection broken: IncompleteRead(9675 bytes read, 565 more expected)', IncompleteRead(9675 bytes read, 565 more expected))


...

Ошибка при парсинге https://db.chgk.info/tour/bit-1: ('Connection broken: IncompleteRead(3531 bytes read, 6709 more expected)', IncompleteRead(3531 bytes read, 6709 more expected))


Парсинг:  93%|█████████▎| 300/322 [13:56<03:50, 10.50s/it]

Сохранено: 5472 турниров


Парсинг: 100%|██████████| 322/322 [15:04<00:00,  2.81s/it]



Готово! Всего: 5494 турниров, 329891 вопросов


In [ ]:
from google.colab import files

for f in ['tournaments.csv', 'authors.csv', 'questions.csv', 'chgk_data.pkl']:
    if os.path.exists(f):
        files.download(f)
        time.sleep(0.5)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
error_links = [
    'https://db.chgk.info/tour/arhang12', 'https://db.chgk.info/tour/spbvuz04',
    'https://db.chgk.info/tour/gele09br', 'https://db.chgk.info/tour/igp9',
    'https://db.chgk.info/tour/kol09v', 'https://db.chgk.info/tour/poltbr03',
    'https://db.chgk.info/tour/cup11brj', 'https://db.chgk.info/tour/irc2003',
    'https://db.chgk.info/tour/mgp1112', 'https://db.chgk.info/tour/korp03-1',
    'https://db.chgk.info/tour/tourde10', 'https://db.chgk.info/tour/korp03-2',
    'https://db.chgk.info/tour/tyumar09', 'https://db.chgk.info/tour/taviv03',
    'https://db.chgk.info/tour/marafo11', 'https://db.chgk.info/tour/elebr03',
    'https://db.chgk.info/tour/marafo10', 'https://db.chgk.info/tour/konsis95',
    'https://db.chgk.info/tour/marafo09', 'https://db.chgk.info/tour/postny94',
    'https://db.chgk.info/tour/ukbr11sh', 'https://db.chgk.info/tour/burda93',
    'https://db.chgk.info/tour/ukbr11st', 'https://db.chgk.info/tour/elebr02',
    'https://db.chgk.info/tour/zelbr10', 'https://db.chgk.info/tour/burda92',
    'https://db.chgk.info/tour/marafo07', 'https://db.chgk.info/tour/ukrbr02',
    'https://db.chgk.info/tour/marafo06', 'https://db.chgk.info/tour/ukbr01sh',
    'https://db.chgk.info/tour/ukbr10st', 'https://db.chgk.info/tour/ukbr02sh',
    'https://db.chgk.info/tour/cup10brj', 'https://db.chgk.info/tour/kop1998',
    'https://db.chgk.info/tour/mgp10910', ' https://db.chgk.info/tour/igp8',
    'https://db.chgk.info/tour/taviv10', ' https://db.chgk.info/tour/taviv01',
    'https://db.chgk.info/tour/kol08o', 'https://db.chgk.info/tour/ostrov99',
    'https://db.chgk.info/tour/haif1l10', 'https://db.chgk.info/tour/ukrbr01',
    'https://db.chgk.info/tour/taviv09', 'https://db.chgk.info/tour/zagadki',
    'https://db.chgk.info/tour/haif1l09', 'https://db.chgk.info/tour/mgp0001',
    'https://db.chgk.info/tour/ukrbr11', 'https://db.chgk.info/tour/mgp9900',
    'https://db.chgk.info/tour/tor0809', 'https://db.chgk.info/tour/kopejka',
    'https://db.chgk.info/tour/ukrbr10', 'https://db.chgk.info/tour/nsk',
    'https://db.chgk.info/tour/ukbr10sh', 'https://db.chgk.info/tour/igp5',
    'https://db.chgk.info/tour/kol0506', 'https://db.chgk.info/tour/igp6o',
    'https://db.chgk.info/tour/kol05v', 'https://db.chgk.info/tour/smcat99',
    'https://db.chgk.info/tour/nizhni05',
    'https://db.chgk.info/tour/nizhni06',
    'https://db.chgk.info/tour/provi06',
    'https://db.chgk.info/tour/ruscup06',
    'https://db.chgk.info/tour/mgtu04',
    'https://db.chgk.info/tour/smol06-2',
    'https://db.chgk.info/tour/kop1995',
    'https://db.chgk.info/tour/kop1996',
    'https://db.chgk.info/tour/genius05',
    'https://db.chgk.info/tour/ukrcup05',
    'https://db.chgk.info/tour/odes0203',
    'https://db.chgk.info/tour/genius02',
    'https://db.chgk.info/tour/genius03',
    'https://db.chgk.info/tour/genius04',
    'https://db.chgk.info/tour/haif1l05',
    'https://db.chgk.info/tour/perlin',
    'https://db.chgk.info/tour/elebr04',
    'https://db.chgk.info/tour/kop1997',
    'https://db.chgk.info/tour/taviv04',
    'https://db.chgk.info/tour/har04v-2',
    'https://db.chgk.info/tour/dneprop',
    'https://db.chgk.info/tour/ukbr08sh',
    'https://db.chgk.info/tour/ukbr08st',
    'https://db.chgk.info/tour/marafo03',
    'https://db.chgk.info/tour/haif1l04',

]

In [20]:
for idx, link in enumerate(tqdm(error_links, desc="Парсинг")):
    result = parse_tournament_page(link, idx + 1)
    if result:
        tournament, questions = result
        tournaments_data.append(tournament)
        questions_data.extend(questions)

    time.sleep(0.3)

    if (idx + 1) % 100 == 0:
        save_data()
        print(f"Сохранено: {len(tournaments_data)} турниров")
save_data()
print(f"\nГотово! Всего: {len(tournaments_data)} турниров, {len(questions_data)} вопросов")

Парсинг:   4%|▎         | 3/83 [00:13<05:55,  4.44s/it]

Ошибка при парсинге https://db.chgk.info/tour/igp9: ('Connection broken: IncompleteRead(3531 bytes read, 6709 more expected)', IncompleteRead(3531 bytes read, 6709 more expected))


Парсинг:   5%|▍         | 4/83 [00:18<06:09,  4.67s/it]

Ошибка при парсинге https://db.chgk.info/tour/kol09v: ('Connection broken: IncompleteRead(3531 bytes read, 6709 more expected)', IncompleteRead(3531 bytes read, 6709 more expected))


Парсинг:   7%|▋         | 6/83 [00:29<06:20,  4.94s/it]

...

Ошибка при парсинге https://db.chgk.info/tour/marafo03: ('Connection broken: IncompleteRead(9675 bytes read, 565 more expected)', IncompleteRead(9675 bytes read, 565 more expected))


Парсинг:  99%|█████████▉| 82/83 [12:41<00:12, 12.86s/it]

Ошибка при парсинге https://db.chgk.info/tour/haif1l04: ('Connection broken: IncompleteRead(9675 bytes read, 565 more expected)', IncompleteRead(9675 bytes read, 565 more expected))


Парсинг: 100%|██████████| 83/83 [12:48<00:00,  9.26s/it]



Готово! Всего: 5521 турниров, 334015 вопросов
